In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import time

In [2]:
data = load_breast_cancer()
X = data.data
y = data.target

print(f"Data loaded: {X.shape[0]} rows, {X.shape[1]} columns")
print(f"Target: 0 = malignant, 1 = benign")
print(f"Class distribution: \n{pd.Series(y).value_counts()}")

Data loaded: 569 rows, 30 columns
Target: 0 = malignant, 1 = benign
Class distribution: 
1    357
0    212
Name: count, dtype: int64


In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [4]:
models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier(),
    "Neural Network": MLPClassifier(max_iter=500),
    "XGBoost": XGBClassifier(eval_metric='logloss'),
    "LightGBM": LGBMClassifier(verbose=-1)
}

In [5]:
results = []

In [6]:
for name, model in models.items():
    print(f"Training {name}...", end=" ", flush=True)

    # Use scaled data for some models
    if name in ["SVM", "KNN", "Neural Network"]:
        X_use = X_train_scaled
    else:
        X_use = X_train

    # Time it
    start = time.time()
    scores = cross_val_score(model, X_use, y_train, cv=5, scoring='accuracy')
    end = time.time()

    results.append({
        "Model": name,
        "Accuracy": scores.mean(),
        "Std": scores.std(),
        "Time (sec)": round(end - start, 2)
    })
    print(f"Done! Accuracy: {scores.mean():.3f}")

Training Logistic Regression... Done! Accuracy: 0.938
Training Decision Tree... Done! Accuracy: 0.914
Training Random Forest... 

C:\Users\vedan\PycharmProjects\PythonProject2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\vedan\PycharmProjects\PythonProject2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-lear

Done! Accuracy: 0.956
Training SVM... Done! Accuracy: 0.976
Training KNN... Done! Accuracy: 0.960
Training Neural Network... Done! Accuracy: 0.980
Training XGBoost... Done! Accuracy: 0.965
Training LightGBM... Done! Accuracy: 0.965


C:\Users\vedan\PycharmProjects\PythonProject2\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\vedan\PycharmProjects\PythonProject2\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\vedan\PycharmProjects\PythonProject2\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\vedan\PycharmProjects\PythonProject2\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\vedan\PycharmProjects\PythonProject2\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X d

In [7]:
df_results = pd.DataFrame(results)
df_results = df_results.sort_values("Accuracy", ascending=False).reset_index(drop=True)
df_results.insert(0, "Rank", df_results.index + 1)

print("\n" + "=" * 60)
print("🏆 MODEL LEADERBOARD 🏆")
print("=" * 60)
print(df_results[["Rank", "Model", "Accuracy", "Time (sec)"]].to_string(index=False))


winner = df_results.iloc[0]
print(f"\n✅ WINNER: {winner['Model']} with {winner['Accuracy']:.3f} accuracy")
print(f"   Training time: {winner['Time (sec)']} seconds")


🏆 MODEL LEADERBOARD 🏆
 Rank               Model  Accuracy  Time (sec)
    1      Neural Network  0.980220        1.84
    2                 SVM  0.975824        0.02
    3             XGBoost  0.964835        0.45
    4            LightGBM  0.964835        0.27
    5                 KNN  0.960440        3.50
    6       Random Forest  0.956044        0.88
    7 Logistic Regression  0.938462        0.15
    8       Decision Tree  0.914286        0.03

✅ WINNER: Neural Network with 0.980 accuracy
   Training time: 1.84 seconds
